<a href="https://colab.research.google.com/github/sidersaurio/labs/blob/main/Wave_Equation_PINN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Wave Equation PINN

### PINN

In [1]:
from collections import OrderedDict
from dataclasses import dataclass
from typing import Callable

import torch
from torch import nn
from torch.func import functional_call, grad, vmap


@dataclass
class Config:
    num_hidden: int = 5
    dim_hidden: int = 5
    batch_size: int = 32
    learning_rate: float = 1e-1
    num_epochs: int = 100


class LinearNN(nn.Module):
    def __init__(
        self,
        num_inputs: int = 1,
        num_layers: int = 1,
        num_neurons: int = 5,
        act: nn.Module = nn.Tanh(),
    ) -> None:
        """Basic neural network architecture with linear layers

        Args:
            num_inputs (int, optional): the dimensionality of the input tensor
            num_layers (int, optional): the number of hidden layers
            num_neurons (int, optional): the number of neurons for each hidden layer
            act (nn.Module, optional): the non-linear activation function to use for stitching
                linear layers togeter
        """
        super().__init__()

        self.num_inputs = num_inputs
        self.num_neurons = num_neurons
        self.num_layers = num_layers

        layers = []

        # input layer
        layers.append(nn.Linear(self.num_inputs, num_neurons))

        # hidden layers with linear layer and activation
        for _ in range(num_layers):
            layers.extend([nn.Linear(num_neurons, num_neurons), act])

        # output layer
        layers.append(nn.Linear(num_neurons, 1))

        # build the network
        self.network = nn.Sequential(*layers)

    def forward(self, *inputs: tuple[torch.Tensor, ...]) -> torch.Tensor:

        if len(inputs) != self.num_inputs:
            raise ValueError(f"Expected {self.num_inputs} inputs, got {len(inputs)}")

        if self.num_inputs == 1:
            input = inputs[0]
            return self.network(input.reshape(-1, 1)).squeeze()
        else:

            # NOTE
            # when receiving vectors from the application of a grad
            # higher-order function, the type of the tensor is slightly
            # different and their shape is not defined. With these tensors,
            # stacking works on dimension 0, thus the definition
            # of the `dim` variable below
            dim = int(all([input.shape for input in inputs]))

            input = torch.stack(inputs, dim=dim)
            return self.network(input).squeeze()


def make_forward_fn_1d(
    model: nn.Module,
    derivative_order: int = 1,
) -> list[Callable]:
    """Make a functional forward pass and gradient functions given an input model in 1-dimension

    This function creates a set of functional calls of the input model

    It returns a list of composable v-mapped version of the forward pass
    and of higher-order derivatives with respect to the inputs as
    specified by the input argument `derivative_order`

    Args:
        model (nn.Module): the model to make the functional calls for. It can be any subclass of
            a nn.Module
        derivative_order (int, optional): Up to which order return functions for computing the
            derivative of the model with respect to the inputs

    Returns:
        list[Callable]: A list of functions where each element corresponds to
            a v-mapped version of the model forward pass and its derivatives. The
            0-th element is always the forward pass and, depending on the value of
            the `derivative_order` argument, the following elements corresponds to
            the i-th order derivative function with respect to the model inputs. The
            vmap ensures efficient support for batched inputs
    """
    # notice that `functional_call` supports batched input by default
    # thus there is not need to call vmap on it, as it's instead the case
    # for the derivative calls
    def f(x: torch.Tensor, params: dict[str, torch.nn.Parameter] | tuple[torch.nn.Parameter, ...]) -> torch.Tensor:

        # the functional optimizer works with parameters represented as a tuple instead
        # of the dictionary form required by the `functional_call` API
        # here we perform the conversion from tuple to dictionary
        if isinstance(params, tuple):
            params_dict = tuple_to_dict_parameters(model, params)
        else:
            params_dict = params

        return functional_call(model, params_dict, (x, ))

    fns = []
    fns.append(f)

    dfunc = f
    for _ in range(derivative_order):

        # first compute the derivative function
        dfunc = grad(dfunc)

        # then use vmap to support batching
        dfunc_vmap = vmap(dfunc, in_dims=(0, None))

        fns.append(dfunc_vmap)

    return fns


def make_forward_fn_nd(
    model: nn.Module,
    on_variable: int,
    derivative_order: int = 1,
) -> list[Callable]:
    """Make a functional forward pass and gradient functions given an input model in n-dimensions.

    The parameters are exactly as the function above. The call to `grad` has been replaced with
    a call to the reversed mode AD `jacrev` which computes the Jacobian of the function. Notice that
    `jacrev` automatically supports batched inputs.
    """

    # notice that `functional_call` supports batched input by default
    def f(*inputs: torch.Tensor, params: dict[str, torch.nn.Parameter] | tuple[torch.nn.Parameter, ...] = None) -> torch.Tensor:
        if isinstance(params, tuple):
            params_dict = tuple_to_dict_parameters(model, params)
        else:
            params_dict = params

        return functional_call(model, params_dict, inputs)

    fns = []
    fns.append(f)

    dfunc = f
    for _ in range(derivative_order):

        dfunc = grad(dfunc, argnums=on_variable)
        dfunc_vmap = vmap(dfunc)

        fns.append(dfunc_vmap)

    return fns


def tuple_to_dict_parameters(
        model: nn.Module, params: tuple[torch.nn.Parameter, ...]
) -> OrderedDict[str, torch.nn.Parameter]:
    """Convert a set of parameters stored as a tuple into a dictionary form

    This conversion is required to be able to call the `functional_call` API which requires
    parameters in a dictionary form from the results of a functional optimization step which
    returns the parameters as a tuple

    Args:
        model (nn.Module): the model to make the functional calls for. It can be any subclass of
            a nn.Module
        params (tuple[Parameter, ...]): the model parameters stored as a tuple

    Returns:
        An OrderedDict instance with the parameters stored as an ordered dictionary
    """
    keys = list(dict(model.named_parameters()).keys())
    values = list(params)
    return OrderedDict(({k:v for k,v in zip(keys, values)}))

### Training

In [10]:
!pip install torchopt

In [2]:
from typing import Callable

import torch
import torchopt
from torch import Tensor


def train_pinn(
    model: torch.nn.Module,
    loss_fn: Callable,
    domain_sampler: Callable,
    learning_rate: float = 1e-3,
    num_iter: int = 100,
) -> tuple[tuple[Tensor, ...], Tensor]:

    print_every = num_iter // min(num_iter, 50)

    # choose optimizer with functional API using functorch
    optimizer = torchopt.FuncOptimizer(torchopt.adam(lr=learning_rate))

    # initial parameters, randomly initialized
    params = tuple(model.parameters())

    # train the model
    loss_evolution = []
    for i in range(num_iter):

        # sample points in the domain for each epoch
        x = domain_sampler()

        # compute the loss with the current parameters
        loss = loss_fn(params, x)

        # update the parameters with functional optimizer
        params = optimizer.step(loss, params)

        if i % print_every == 0:
            print(f"Iteration {i} with loss {float(loss)}")

        loss_evolution.append(float(loss))

    return params, loss_evolution

### Plotting

In [3]:

from typing import Callable

import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.animation import FuncAnimation
import torch
from torch import Tensor


def plot_1d_solution(
    x_eval: Tensor,
    x_sample: Tensor,
    f_eval: Tensor,
    analytical_sol_fn: Callable,
    loss_evolution: Tensor,
    show: bool = True,
) -> Figure:
    """
    Plot the solution of a 1-dimension differential equation

    Args:
        x_eval (torch.Tensor): Evaluation points for the solution.
        f_eval (torch.Tensor): Evaluated solution at the evaluation points.
        analytical_sol_fn (callable): Analytical solution function.
        x_sample_np (numpy.ndarray): Sample training points.
        loss_evolution (list): List of loss values at each epoch.
    """
    x_eval_np = x_eval.detach().numpy()
    x_sample_np = x_sample.detach().numpy()

    fig, ax = plt.subplots()

    ax.scatter(x_sample_np, analytical_sol_fn(x_sample_np), color="red", label="Sample training points")
    ax.plot(x_eval_np, f_eval.detach().numpy(), label="PINN final solution")
    ax.plot(
        x_eval_np,
        analytical_sol_fn(x_eval_np),
        label="Analytic solution",
        color="green",
        alpha=0.75,
    )
    ax.set(title="Equation solved with NNs", xlabel="t", ylabel="f(t)")
    ax.legend()

    fig, ax = plt.subplots()
    ax.semilogy(loss_evolution)
    ax.set(title="Loss evolution", xlabel="# epochs", ylabel="Loss")
    ax.legend()

    if show:
        plt.show()

    return fig


def animate_2d_solution(
    x_eval: Tensor,
    t_eval: Tensor,
    opt_params: tuple,
    fn: Callable,
    show: bool = True
) -> tuple[Figure, FuncAnimation]:
    """
    Animate the solution of a 2-dimension problem in time and space

    Args:
        x_eval (Tensor): Evaluation points for the spatial dimension.
        t_eval (Tensor): Evaluation points for the time dimension.
        opt_params (tuple): Optimal parameters computed after the training procedure.
        fn (Callable): The function to animate.
        show (bool, optional): Whether to display the animation. Defaults to True.

    Returns:
        tuple[Figure, FuncAnimation]: A tuple containing the matplotlib Figure and FuncAnimation objects.
    """
    fig, ax = plt.subplots()
    line, = ax.plot([], [], lw=2)

    ax.set(title="Equation solved with NNs", xlabel="x", ylabel="u(x,t)")

    def init() -> tuple:
        ax.set_xlim(x_eval[0].item(), x_eval[-1].item())
        y_values = [fn(x_eval, t * torch.ones_like(x_eval), params=opt_params).detach().numpy() for t in t_eval]
        ax.set_ylim(min(map(min, y_values)), max(map(max, y_values)))
        return line,

    def animate(frame: int) -> tuple:
        t = t_eval[frame]
        y = fn(x_eval, t * torch.ones_like(x_eval), params=opt_params)
        line.set_data(x_eval.detach().numpy(), y.detach().numpy())
        return line,

    ani = FuncAnimation(
        fig, animate, frames=len(t_eval), init_func=init, blit=True, interval=200, repeat=False
    )

    if show:
        plt.show()

    return fig, ani

### Wave Equation

In [ ]:
from typing import Callable
import argparse

import torch
from torch import nn
'''
from pinn import make_forward_fn_nd, LinearNN, Config
from training import train_pinn
from plotting import animate_2d_solution
'''
C = 1.0  # wave speed

# initial and final positions where the string is attached
X_I = 0.
X_F = torch.pi

# boundary conditions for the displacement
F_0 = 0.  # displacement at the beginning of the string
F_L = 0.  # displacement at the end of the string

# boundary conditions for the elapsed time
T_I = 0.  # initial time
T_F = 4.  # final time


def icond_u(x: torch.Tensor) -> torch.Tensor:
    """Initial condition of the wave equation displacement

    Mathematical formulation: u(x, t = 0) = sin(x)
    """
    return torch.sin(x)


def icond_ut(x: torch.Tensor) -> torch.Tensor:
    """Initial condition of the wave equation velocity

    Mathematical formulation: du/dt(x, t = 0) = 0
    """
    return torch.zeros_like(x)


def make_loss_fn(
    f: Callable,
    dfdt: Callable,
    d2fdx2: Callable,
    d2fdt2: Callable
) -> Callable:
    """Make a function loss evaluation function

    The loss is computed as sum of the interior MSE loss (the differential equation residual)
    and the MSE of the loss at the boundary

    Args:
        f (Callable): The functional forward pass of the model used a universal function approximator. This
            is a function with signature (x, params) where `x` is the input data and `params` the model
            parameters
        dfdt (Callable): First order pure partial derivative w.r.t. t. This is computed via the functional
            `grad` higher-order function and it supports batches
        d2fdx2 (Callable): Second order pure partial derivative w.r.t. x. This is computed via repeated application
            of the functional `grad` higher-order function and it supports batches
        d2fdt2 (Callable): Second order pure partial derivative w.r.t. t. This is computed via repeated application
            of the functional `grad` higher-order function and it supports batches

    Returns:
        Callable: The loss function with signature (params, x) where `x` is the input data and `params` the model
            parameters. Notice that a simple call to `dloss = functorch.grad(loss_fn)` would give the gradient
            of the loss with respect to the model parameters needed by the optimizers
    """

    def loss_fn(params: torch.Tensor, input: torch.Tensor | tuple[torch.Tensor, ...]):

        t, x = input

        # interior loss
        interior = d2fdt2(x, t, params=params) - (C ** 2) * d2fdx2(x, t, params=params)

        # boundary conditions
        x0_val = torch.ones_like(x) * X_I
        f0_val = torch.ones_like(x) * F_0
        boundary_x0 = f(x0_val, t, params=params) - f0_val

        xL_val = torch.ones_like(x) * X_F
        fL_val = torch.ones_like(x) * F_L
        boundary_xL = f(xL_val, t, params=params) - fL_val

        # initial condition on displacement
        t_initial = torch.ones_like(t) * T_I
        f_initial = icond_u(x)
        f_initial_val = f(x, t_initial, params=params) - f_initial

        # initial condition on velocity
        dfdt_initial = icond_ut(x)
        dfdt_initial_val = dfdt(x, t_initial, params=params) - dfdt_initial

        loss = nn.MSELoss()
        loss_value = \
            loss(interior, torch.zeros_like(interior)) + \
            loss(boundary_x0, torch.zeros_like(boundary_x0)) + \
            loss(boundary_xL, torch.zeros_like(boundary_xL)) + \
            loss(f_initial_val, torch.zeros_like(f_initial_val)) + \
            loss(dfdt_initial_val, torch.zeros_like(dfdt_initial_val))

        return loss_value

    return loss_fn


if __name__ == "__main__":

    # make it reproducible
    torch.manual_seed(42)

    # parse input from user
    parser = argparse.ArgumentParser()

    parser.add_argument("-n", "--num-hidden", type=int, default=5)
    parser.add_argument("-d", "--dim-hidden", type=int, default=5)
    parser.add_argument("-b", "--batch-size", type=int, default=32)
    parser.add_argument("-lr", "--learning-rate", type=float, default=1e-1)
    parser.add_argument("-e", "--num-epochs", type=int, default=10_000)

    args = parser.parse_args(args=[])
    config = Config(**args.__dict__)

    domain_x = (X_I, X_F)
    domain_t = (T_I, T_F)

    def domain_sampler() -> tuple[torch.Tensor, torch.Tensor]:
        x = torch.FloatTensor(config.batch_size).uniform_(domain_x[0], domain_x[1])
        t, _ = torch.sort(torch.FloatTensor(config.batch_size).uniform_(domain_t[0], domain_t[1]))
        t_and_x = torch.cartesian_prod(t, x)
        return t_and_x[:, 0], t_and_x[:, 1]

    # MLP model
    model = LinearNN(num_layers=config.num_hidden, num_neurons=config.dim_hidden, num_inputs=2)

    f, dfdt, d2fdt2 = make_forward_fn_nd(model, on_variable=0, derivative_order=2)
    _, _, d2fdx2 = make_forward_fn_nd(model, on_variable=1, derivative_order=2)

    loss_fn = make_loss_fn(f, dfdt, d2fdx2, d2fdt2)

    inputs = domain_sampler()
    initial_params = tuple(model.parameters())
    initial_loss = loss_fn(initial_params, inputs)
    print(f"Initial loss: {initial_loss.item()}")

    opt_params, loss_evolution = train_pinn(
        model,
        loss_fn,
        domain_sampler,
        learning_rate=config.learning_rate,
        num_iter=config.num_epochs,
    )

    x_eval = torch.arange(domain_x[0], domain_x[1], 0.01)
    t_eval = torch.arange(domain_t[0], domain_t[1], 0.1)

    _, ani = animate_2d_solution(x_eval, t_eval, opt_params, f, show=True)

    ani.save("wave_equation_1d.gif", writer="pillow")

Initial loss: 1.3258181810379028
Iteration 0 with loss 1.1106773614883423


/tmp/ipykernel_9633/2192521157.py:38: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"Iteration {i} with loss {float(loss)}")


Iteration 200 with loss 0.1831463724374771
Iteration 400 with loss 0.1414594203233719
Iteration 600 with loss 0.1783689260482788
Iteration 800 with loss 0.16896802186965942
Iteration 1000 with loss 0.1467665284872055
